# M3 · Less computation — Turbo 8 steps vs baseline 40

**Learning objective:** measure a *less computation* candidate (Turbo distilled DiT, 8-step schedule) against the B0 baseline on your W7900, compare observed speedup with an Amdahl prediction, and reason honestly about the **confound**: B0→L1 changes the model variant AND the sampling schedule, not just a step-count knob.

**Predict first:** naive step ratio 40/8 = 5×. Will you measure 5×? Write your prediction and your reason.

Reference lesson: [M03 lesson](../course/M03_less_computation/LESSON.md) · [中文](./03_less_computation.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="03")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 1 · The confound, stated up front
B0 = full BF16 DiT, 40 steps. L1 = Turbo BF16 DiT, 8 steps. A *clean* step-count experiment would fix the model and vary only steps (e.g. Turbo at 8 vs 40 steps, same sampler/scheduler). This lab measures the **deployable configuration pair**; keep the distinction explicit in anything you report.

In [ ]:
CONF0UND_NOTE = ("B0->L1 changes model variant + schedule + steps; "
    "it is NOT a single-variable step-count experiment")
print(CONF0UND_NOTE)
ref = ns.summary_rows()
for k in ("B0", "L1"):
    r = ref[k]
    _stamp(HISTORICAL, f"{k}: median {r['warm_total_median_s']}s n={r['n_measure']}")

## 2 · Run both arms on YOUR W7900
Baseline B0 (bounded or full, matching your mode) then candidate L1. If you already produced a B0 in M1 in this campaign it is reused with its run_id shown — records are append-only, waste is not repeated.

In [ ]:
def run_less_pair():
    prof = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    b0 = session.run("B0", profile=prof)
    l1 = session.run("L1", profile=prof)
    return b0, l1

pair = gpu_cell(run_less_pair) if want_own_arms() else None
pair_origin = "CURRENT_SESSION_PHYSICAL"
if pair is None:
    b0_id = session.latest_campaign_run("B0")
    l1_id = session.latest_campaign_run("L1")
    if b0_id and l1_id:
        pair = (session.consume(b0_id, why="explicit reuse (shown run_id): B0"),
                session.consume(l1_id, why="explicit reuse (shown run_id): L1"))
        pair_origin = session.consumed[-1].get("category", "COMMITTED_PHYSICAL_REFERENCE")
if pair is not None:
    b0b, l1b = pair
    ab = rt.ab_compare(b0b, l1b, label="B0 (40 steps) vs L1 (Turbo 8 steps)")
    print({k: ab[k] for k in ("classification", "speedup_median",
                              "latency_delta_s", "vram_delta_mb")})

## 3 · Predicted vs observed speedup
Amdahl with the DiT share `d` from your M2 pipeline decomposition (or the reference 0.83): predicted = 1/(d·(8/40) + (1−d)). Compare with the measured ratio and explain the residual.

In [ ]:
if pair is not None:
    b0b, l1b = pair
    bs, ls = b0b.stats(), l1b.stats()
    d = 0.83  # DiT share of warm latency (replace with YOUR M2 value)
    predicted = 1.0 / (d * (8 / 40) + (1 - d))
    measured = bs["median_s"] / ls["median_s"]
    b0_steps = (bs.get("steps") or 40) or 40
    l1_steps = (ls.get("steps") or 8) or 8
    per_step_b0 = bs["median_s"] / b0_steps
    per_step_l1 = ls["median_s"] / l1_steps
    print(f"predicted (Amdahl, d={d}): {predicted:.2f}x   measured: {measured:.2f}x")
    print(f"per-step: B0 {per_step_b0:.3f}s  L1 {per_step_l1:.3f}s "
          f"(ratio {per_step_b0/per_step_l1:.2f}x)")
    _stamp(_origin_stamp(pair_origin),
           f"measured speedup {measured:.2f}x [{ab['classification']}]")
    _stamp(ESTIMATE, f"Amdahl prediction {predicted:.2f}x (derived from model share)")
    print("limitation: Turbo changes the model too; per-step costs differ; "
          "quality is NOT addressed by this comparison (see section 4)")

## 4 · Quality is a separate axis
Less sampling is not free: Turbo changes output statistics. The course gate scores L1 vs B0 with **distance semantics** (LPIPS as a perceptual *distance*, not a fidelity score) plus human review of paired grids. Never claim Turbo is 'equivalent' to the 40-step baseline.

> **Quality decisions stay human.** No cell auto-approves an image-quality verdict. Historical owner verdicts apply **only** to their originally evaluated deployment scope; a new physical run needs its own review (`PENDING_HUMAN_REVIEW` until then).

In [ ]:
qs = ns.summary_rows().get("L1", {})
print("frozen campaign L1 quality verdict:",
      qs.get("quality_verdict", "see evidence/quality/L1/"))
print("paired grids + native-res poster crops: evidence/quality/L1/")
print("in GPU modes run your own gate:  session.run_quality('L1', reference='B0')")
print("your verdict on YOUR images stays PENDING_HUMAN_REVIEW until you review the grid")

## 5 · Your turn — design the clean experiment
Specify a controlled step-count experiment: fixed Turbo BF16 model + sampler, steps swept 8/16/40, seeds and prompts frozen. What remains incomparable to B0 even then? (model capacity, distillation, CFG=1 schedule...)

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment L1 --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.